# 제목 
Learning Face - Mini Study Coach with LangGraph

#그래프 구조 
START → analyze_request → make_study_plan → make_quiz → END

In [1]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool

In [2]:
#tool 만들기 
@tool
def learning_tip_tool(topic: str, level: str) -> str:
    """주제와 난이도에 맞는 간단한 학습 팁을 제공합니다."""
    if level == "beginner":
        return f"{topic}을 공부할 때는 먼저 쉬운 예제를 따라 해보고, 직접 한 번 바꿔보는 것이 좋습니다."
    return f"{topic}은 개념을 복습한 뒤 응용 문제를 풀면서 실력을 높이는 것이 좋습니다."

In [3]:
# State 정의
class StudyState(TypedDict):
    user_input: str
    topic: str
    request_type: str
    level: str
    plan: List[str]
    quiz: List[str]
    tip: str
    final_output: str

In [4]:
# Node 1: 사용자 요청 분석
def analyze_request(state: StudyState):
    user_input = state["user_input"]

    topic = user_input
    request_type = "plan"
    level = "beginner"

    if "퀴즈" in user_input:
        request_type = "quiz"

    if "중급" in user_input:
        level = "intermediate"
    elif "초보" in user_input:
        level = "beginner"

    return {
        "topic": topic,
        "request_type": request_type,
        "level": level,
    }

In [5]:
# Node 2: 학습 계획 생성
def make_study_plan(state: StudyState):
    topic = state["topic"]
    level = state["level"]

    if level == "beginner":
        plan = [
            f"{topic}의 기본 개념 이해하기",
            f"{topic}의 쉬운 예제 따라 하기",
            f"{topic} 관련 짧은 연습문제 풀기"
        ]
    else:
        plan = [
            f"{topic} 핵심 개념 복습하기",
            f"{topic} 실전 예제 분석하기",
            f"{topic} 응용 문제 풀기"
        ]

    return {
        "plan": plan,
        "quiz": []
    }

In [6]:
# Node 3: 복습 질문 생성
def make_quiz(state: StudyState):
    topic = state["topic"]

    quiz = [
        f"{topic}에서 가장 중요한 개념은 무엇인가요?",
        f"{topic}를 실제로 사용할 수 있는 예시는 무엇인가요?",
        f"{topic}를 배울 때 주의할 점은 무엇인가요?"
    ]

    return {
        "quiz": quiz,
        "plan": []
    }

In [7]:
# tool 사용 
def add_learning_tip(state: StudyState):
    topic = state["topic"]
    level = state["level"]

    tip = learning_tip_tool.invoke({"topic": topic, "level": level})

    return {
        "tip": tip
    }

In [8]:
# 최종 응답 정리 
def final_response(state: StudyState):
    request_type = state["request_type"]
    tip = state["tip"]

    if request_type == "plan":
        plan_text = "\n".join([f"- {item}" for item in state["plan"]])
        final_output = f"""[학습 계획]
{plan_text}

[학습 팁]
{tip}
"""
    else:
        quiz_text = "\n".join([f"- {item}" for item in state["quiz"]])
        final_output = f"""[퀴즈]
{quiz_text}

[학습 팁]
{tip}
"""

    return {
        "final_output": final_output
    }

In [9]:
# Conditional Edge 함수
def route_by_request(state: StudyState):
    if state["request_type"] == "quiz":
        return "make_quiz"
    return "make_study_plan"

In [10]:
# 그래프 만들기
builder = StateGraph(StudyState)

builder.add_node("analyze_request", analyze_request)
builder.add_node("make_study_plan", make_study_plan)
builder.add_node("make_quiz", make_quiz)
builder.add_node("add_learning_tip", add_learning_tip)
builder.add_node("final_response", final_response)

builder.add_edge(START, "analyze_request")

builder.add_conditional_edges(
    "analyze_request",
    route_by_request,
    {
        "make_study_plan": "make_study_plan",
        "make_quiz": "make_quiz"
    }
)

builder.add_edge("make_study_plan", "add_learning_tip")
builder.add_edge("make_quiz", "add_learning_tip")
builder.add_edge("add_learning_tip", "final_response")
builder.add_edge("final_response", END)

graph = builder.compile()

In [11]:
# 실행 예시 1 
result = graph.invoke({
    "user_input": "파이썬 if문 초보 공부 계획 만들어줘",
    "topic": "",
    "request_type": "",
    "level": "",
    "plan": [],
    "quiz": [],
    "tip": "",
    "final_output": ""
})

print(result["final_output"])

[학습 계획]
- 파이썬 if문 초보 공부 계획 만들어줘의 기본 개념 이해하기
- 파이썬 if문 초보 공부 계획 만들어줘의 쉬운 예제 따라 하기
- 파이썬 if문 초보 공부 계획 만들어줘 관련 짧은 연습문제 풀기

[학습 팁]
파이썬 if문 초보 공부 계획 만들어줘을 공부할 때는 먼저 쉬운 예제를 따라 해보고, 직접 한 번 바꿔보는 것이 좋습니다.



In [12]:
# 실행 예시 2
result2 = graph.invoke({
    "user_input": "파이썬 if문 퀴즈 내줘",
    "topic": "",
    "request_type": "",
    "level": "",
    "plan": [],
    "quiz": [],
    "tip": "",
    "final_output": ""
})

print(result2["final_output"])

[퀴즈]
- 파이썬 if문 퀴즈 내줘에서 가장 중요한 개념은 무엇인가요?
- 파이썬 if문 퀴즈 내줘를 실제로 사용할 수 있는 예시는 무엇인가요?
- 파이썬 if문 퀴즈 내줘를 배울 때 주의할 점은 무엇인가요?

[학습 팁]
파이썬 if문 퀴즈 내줘을 공부할 때는 먼저 쉬운 예제를 따라 해보고, 직접 한 번 바꿔보는 것이 좋습니다.

